In [1]:
# 필요한 라이브러리 설치
!pip install faker bcrypt

In [2]:
from faker import Faker
import random
import bcrypt
from datetime import datetime, timedelta

fake = Faker('ko_KR')
print("라이브러리 로드 완료")

라이브러리 로드 완료


In [3]:
# ── 생성할 회원 수 설정 ──
USER_COUNT = 100

In [4]:
def generate_password_hash(password: str) -> str:
    return bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt()).decode('utf-8')

def random_datetime(start_year=2022, end_year=2025):
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    return start + timedelta(seconds=random.randint(0, int((end - start).total_seconds())))

def generate_users(count: int) -> list:
    users = []
    password_hash = generate_password_hash('password1234')
    for i in range(1, count + 1):
        name             = fake.name()
        login_id         = f"user{i:04d}"
        email            = fake.unique.email()
        phone            = fake.phone_number()
        role             = 'USER'
        created_at       = random_datetime()
        is_active        = random.choices([1, 0], weights=[85, 15])[0]
        marketing_agreed = random.choice([True, False])
        users.append({
            'name':             name,
            'login_id':         login_id,
            'password':         password_hash,
            'email':            email,
            'phone':            phone,
            'role':             role,
            'created_at':       created_at.strftime('%Y-%m-%d %H:%M:%S'),
            'updated_at':       created_at.strftime('%Y-%m-%d %H:%M:%S'),
            'is_active':        is_active,
            'marketing_agreed': marketing_agreed,
        })
    return users

def to_sql_users(users: list) -> str:
    lines = []
    lines.append(
        "INSERT INTO users "
        "(name, login_id, password, email, phone, role, created_at, updated_at, is_active, marketing_agreed) VALUES"
    )
    rows = []
    for u in users:
        name     = u['name'].replace("'", "''")
        email    = u['email'].replace("'", "''")
        phone    = u['phone'].replace("'", "''")
        password = u['password'].replace("'", "''")
        rows.append(
            f"  ('{name}', '{u['login_id']}', '{password}', '{email}', "
            f"'{phone}', '{u['role']}', '{u['created_at']}', '{u['updated_at']}', "
            f"{u['is_active']}, {str(u['marketing_agreed']).upper()})"
        )
    lines.append(',\n'.join(rows) + ';')
    return '\n'.join(lines)

print("함수 정의 완료")

함수 정의 완료


In [5]:
print(f"회원 {USER_COUNT}명 생성 중...")
users = generate_users(USER_COUNT)
sql   = to_sql_users(users)

# ── 저장 경로 설정 (필요시 변경) ──
output_path = 'users_data.sql'

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(sql)

print(f"완료: {output_path} 생성됨 ({USER_COUNT}명)")

회원 100명 생성 중...
완료: users_data.sql 생성됨 (100명)


In [6]:
print("\n샘플 3개:")
for u in users[:3]:
    print(f"  이름: {u['name']}, login_id: {u['login_id']}, email: {u['email']}, phone: {u['phone']}, is_active: {u['is_active']}, marketing_agreed: {u['marketing_agreed']}")


샘플 3개:
  이름: 박정웅, login_id: user0001, email: yeeunmin@example.net, phone: 070-4282-7541, is_active: 1, marketing_agreed: False
  이름: 김영숙, login_id: user0002, email: ngim@example.net, phone: 054-400-4668, is_active: 1, marketing_agreed: True
  이름: 송순자, login_id: user0003, email: jihuhan@example.com, phone: 010-8006-5066, is_active: 1, marketing_agreed: False
